# 02. Field Constraints & Inspecting ValidationError

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/en/02_fields_and_errors.ipynb)

Type annotations like `int` and `str` guarantee basic shapes, but production payloads also carry domain invariants—positive quantities, bounded SKU formats, closed shipping tiers, isolated default lists, and camelCase JSON keys. Pydantic's `Field(...)`, `typing.Annotated`, and `typing.Literal` attach these rules declaratively onto each attribute, while Pydantic's Rust core (`pydantic-core`) evaluates **all** fields in a single pass—collecting every violation at once instead of halting on the first bad field.

![Figure 2-1: Declarative Field Constraints and Single-Pass Multi-Error Collection](https://examples.drawlib.com/pydantic/images/02_fields_and_errors/ch02_field_constraints.png)

*Figure 2-1: Declarative Field Constraints and Single-Pass Multi-Error Collection*

---

## 1. Declarative Constraints with `Field` & `Literal`

Use `Field(...)` to enforce numeric bounds (`gt`, `ge`, `lt`, `le`), string length caps (`min_length`, `max_length`), and regular expressions (`pattern`). When a constrained type is reused across models, wrap it in `Annotated[T, Field(...)]`; for finite sets of allowed values, use `Literal[...]`.

Notice `tags: list[str] = Field(default_factory=list)` below: in standard Python functions or classes, sharing a mutable default list risks leaking items across instances, whereas `default_factory=list` invokes `list()` freshly every time a new `OrderItem` is created:

In [ ]:
!pip install -q pydantic

In [ ]:
from typing import Annotated, Literal
from pydantic import BaseModel, ConfigDict, Field

SkuCode = Annotated[str, Field(pattern=r"^[A-Z]{3}-\d{4}$")]


class OrderItem(BaseModel):
    sku: SkuCode = "ORD-1024"
    name: str = Field(min_length=3, max_length=40)
    quantity: int = Field(gt=0, le=1000)
    unit_price: float = Field(ge=0.0)
    tier: Literal["standard", "express"] = "standard"
    tags: list[str] = Field(default_factory=list)


item = OrderItem(
    sku="PRO-4090",
    name="Mechanical Keyboard",
    quantity=2,
    unit_price=129.50,
    tier="express",
)
item.tags.append("fragile")

second_item = OrderItem(name="USB-C Cable", quantity=1, unit_price=12.00)
print("Validated item:", item.model_dump())
print("Second item tags (isolated list):", second_item.tags)

External REST and GraphQL APIs frequently exchange `camelCase` JSON keys (`itemQty`, `unitPrice`), whereas Python codebases follow `snake_case` (`item_qty`, `unit_price`). Pairing `Field(alias=...)` with `ConfigDict(populate_by_name=True)` lets your model accept **either** naming convention on input and export `camelCase` payloads on demand via `model_dump(by_alias=True)`:

In [ ]:
class ApiOrderItem(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    item_qty: int = Field(gt=0, le=1000, alias="itemQty")
    unit_price: float = Field(ge=0.0, alias="unitPrice")


# 1. Validate external camelCase JSON payload
from_json = ApiOrderItem.model_validate({"itemQty": 4, "unitPrice": 19.95})

# 2. Construct in Python using idiomatic snake_case names
from_python = ApiOrderItem(item_qty=2, unit_price=49.00)

print("Python snake_case dump:", from_json.model_dump())
print("External camelCase dump:", from_python.model_dump(by_alias=True))

---

## 2. Anatomy of `ValidationError` (`err.errors()`)

Unlike imperative `if ... raise ValueError(...)` checks that abort on the very first bad field, Pydantic validates every field in one pass and bundles all failures into a single `ValidationError`. Calling `err.errors()` returns a structured `list[dict]` where each entry pinpoints `loc`, `type`, `msg`, and `input`—making it ready to serialize directly into HTTP `422 Unprocessable Entity` responses:

![Figure 2-2: Structure of ValidationError.errors() Entries for API Error Responses](https://examples.drawlib.com/pydantic/images/02_fields_and_errors/ch02_validation_error_anatomy.png)

*Figure 2-2: Structure of ValidationError.errors() Entries for API Error Responses*

Let's pass an intentionally invalid payload containing **three simultaneous violations** (`name=""`, `quantity=0`, and `tier="overnight"`), catch `ValidationError`, inspect `err.error_count()`, and extract `(loc, type, msg)` tuples ready for client-facing UI forms or API `422` error payloads:

In [ ]:
from pydantic import ValidationError

try:
    OrderItem(
        name="",
        quantity=0,
        unit_price=29.99,
        tier="overnight",
    )
except ValidationError as err:
    print(f"Collected {err.error_count()} field errors in a single pass:")
    for e in err.errors():
        print((e["loc"], e["type"], e["msg"]))

    api_422_body = {
        "status": 422,
        "errors": [
            {
                "field": ".".join(str(part) for part in e["loc"]),
                "code": e["type"],
                "message": e["msg"],
                "input": e["input"],
            }
            for e in err.errors()
        ],
    }
    print("\nAPI 422 response payload:", api_422_body)